# Ba-133 direct-light photons to R7378A PMT signals

This notebook folds the KingCRAB per-event S1/S2 photon counts through the digitized Hamamatsu R7378A datasheet response. It estimates photoelectrons, anode charge, mean current, triggerable event rates, and an illustrative 50-ohm waveform.

**Important:** argon scintillation is taken as 128 nm, below the first digitized datasheet points (about 142--145 nm). The 128 nm results are extrapolations, not measurements. A real PMT entrance window can also suppress 128 nm light; `WINDOW_TRANSMISSION` is therefore an explicit input.

In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

from DATA.crab_config import CRABRunConfig
from SRC.KingCRAB.raytrace import emission_wavelength_nm
RUN_CONFIG = CRABRunConfig(gas='argon', pressure_bar=1.0)
from SRC.KingCRAB.digitized import load_digitized_datasets, log_interpolate, log_linear_extrapolation, as_plot_datasets as load_webplotdigitizer_json

from pathlib import Path
import io
import json
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fundamental charge in coulombs.
E_CHARGE_C = 1.602176634e-19

# Locate the project whether Jupyter starts in the repository or CODE folder.
PROJECT = Path.cwd().resolve()
if not (PROJECT / 'DATA').exists() and (PROJECT.parent / 'DATA').exists():
    PROJECT = PROJECT.parent

DATA = PROJECT / 'DATA'
print(f'Project: {PROJECT}')

## User settings

Change these values for the actual PMT bias, optical interface, and oscilloscope. A transmission of 1 represents the ideal no-window calculation; it does **not** claim that the physical R7378A window transmits 128 nm.

In [ ]:
ARGON_WAVELENGTH_NM = emission_wavelength_nm(RUN_CONFIG.gas, RUN_CONFIG.pressure_bar)
PMT_VOLTAGE_V = 900.0
WINDOW_TRANSMISSION = 1.0
COLLECTION_EFFICIENCY = 1.0
LOAD_OHM = 50.0
SPE_FWHM_NS = 8.0
BASELINE_RMS_MV = 0.299  # Terminal 4 auto-trigger result.
TRIGGER_SIGMA = 5.0
RANDOM_SEED = 133

# Number of lowest-wavelength points used for each log-linear extrapolation.
N_LOW_POINTS = 8

# Change this if the downloaded archive has another name or location.
BA_ZIP = Path.home() / 'Downloads' / 'KingCRAB_Ba133_356keV_Ar1bar_DirectRate_direct_rate.zip'
if not BA_ZIP.exists():
    raise FileNotFoundError(f'Could not find Ba-133 analysis archive: {BA_ZIP}')

In [ ]:
spectral = load_digitized_datasets(DATA / 'R7378A_Spectral_Response.json')
characteristics = load_digitized_datasets(DATA / 'R7378A_Characteristics.json')

qe_128_direct, qe_fit, qe_slope, qe_intercept = log_linear_extrapolation(
    spectral['Quantum Efficiency'], ARGON_WAVELENGTH_NM, N_LOW_POINTS, return_fit=True
)
s_128_mA_W, s_fit, s_slope, s_intercept = log_linear_extrapolation(
    spectral['Cathode Radiant Sensitivity'], ARGON_WAVELENGTH_NM, N_LOW_POINTS, return_fit=True
)

# From S = QE * e/(h nu): QE[%] = 124 * S[mA/W] / wavelength[nm].
qe_128_from_sensitivity = 124.0 * s_128_mA_W / ARGON_WAVELENGTH_NM
qe_cases_percent = {
    'low': min(qe_128_direct, qe_128_from_sensitivity),
    'central': np.sqrt(qe_128_direct * qe_128_from_sensitivity),
    'high': max(qe_128_direct, qe_128_from_sensitivity),
}

print(f'Direct QE extrapolation: {qe_128_direct:.3g}%')
print(f'Radiant sensitivity extrapolation: {s_128_mA_W:.3g} mA/W')
print(f'QE derived from sensitivity: {qe_128_from_sensitivity:.3g}%')
print('QE cases:', qe_cases_percent)

In [ ]:
# Plot the measured points and make the extrapolated region unmistakable.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, values, fit, slope, intercept, title, ylabel in [
    (axes[0], spectral['Quantum Efficiency'], qe_fit, qe_slope, qe_intercept, 'Quantum efficiency', 'QE (%)'),
    (axes[1], spectral['Cathode Radiant Sensitivity'], s_fit, s_slope, s_intercept, 'Cathode radiant sensitivity', 'mA/W'),
]:
    wave = np.linspace(ARGON_WAVELENGTH_NM, fit.wavelength_nm.max(), 200)
    ax.scatter(values[:, 0], values[:, 1], s=16, label='Digitized datasheet')
    ax.scatter(fit.wavelength_nm, fit.response, s=34, label='Points used in fit')
    ax.plot(wave, np.exp(intercept + slope * wave), '--', label='Log-linear fit/extrapolation')
    ax.axvline(ARGON_WAVELENGTH_NM, color='black', alpha=0.6, label='Argon: 128 nm')
    ax.axvspan(ARGON_WAVELENGTH_NM, values[:, 0].min(), color='tab:red', alpha=0.1, label='Unmeasured region')
    ax.set(xlabel='Wavelength (nm)', ylabel=ylabel, title=title, xlim=(120, 210))
    ax.set_yscale('log')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Datasheet gain

The digitized `Gain` trace is stored in the JSON on a sensitivity-like scale. The existing King-CRAB characterization interprets the plotted values as gain after multiplication by $10^{13}$. The interpolation below follows that existing convention and uses logarithmic interpolation because PMT gain changes approximately exponentially with voltage.

In [ ]:
gain_trace = characteristics['Gain'].copy()
gain_trace[:, 1] *= 1.0e13

# Interpolate log(gain) at the requested PMT voltage.
if not gain_trace[:, 0].min() <= PMT_VOLTAGE_V <= gain_trace[:, 0].max():
    raise ValueError('PMT voltage lies outside the digitized datasheet gain curve.')
datasheet_gain = float(np.exp(np.interp(PMT_VOLTAGE_V, gain_trace[:, 0], np.log(gain_trace[:, 1]))))
spe_charge_C = E_CHARGE_C * datasheet_gain
spe_area_V_s = LOAD_OHM * spe_charge_C
spe_area_mV_ns = spe_area_V_s * 1.0e12

print(f'Datasheet gain at {PMT_VOLTAGE_V:.0f} V: {datasheet_gain:,.0f}')
print(f'One-photoelectron anode charge: {spe_charge_C:.3e} C')
print(f'One-photoelectron area into {LOAD_OHM:.0f} ohm: {spe_area_mV_ns:.3f} mV ns')

In [ ]:
# Read the million-event CSV directly from the downloaded zip file.
with zipfile.ZipFile(BA_ZIP) as archive:
    csv_name = next(name for name in archive.namelist() if name.endswith('per_event_direct_light.csv'))
    events = pd.read_csv(io.BytesIO(archive.read(csv_name)))

# The line rate already includes the measured source activity and 0.6205 branching fraction.
SOURCE_ACTIVITY_BQ = 12924.1
BA133_356KEV_PROBABILITY = 0.6205
GAMMA_RATE_HZ = SOURCE_ACTIVITY_BQ * BA133_356KEV_PROBABILITY
SIMULATED_GAMMAS = len(events)

rng = np.random.default_rng(RANDOM_SEED)
rows = []
sampled_pe = {}
for case, qe_percent in qe_cases_percent.items():
    pde = np.clip(qe_percent / 100.0 * WINDOW_TRANSMISSION * COLLECTION_EFFICIENCY, 0.0, 1.0)
    for light in ['S1', 'S2']:
        photons = events[f'weighted_{light}_photons_at_plane'].to_numpy()
        # Current simulation weights are unity; rounding makes explicit photon trials.
        photon_trials = np.rint(photons).astype(np.int64)
        pe = rng.binomial(photon_trials, pde)
        sampled_pe[(case, light)] = pe
        event_probability = 1.0 - np.power(1.0 - pde, photon_trials)
        photon_rate = photons.mean() * GAMMA_RATE_HZ
        pe_rate = photon_rate * pde
        trigger_rate = event_probability.mean() * GAMMA_RATE_HZ
        rows.append({
            'case': case, 'light': light, 'QE_percent': qe_percent, 'effective_PDE': pde,
            'incident_photons_per_s': photon_rate, 'photoelectrons_per_s': pe_rate,
            'events_with_at_least_1_PE_per_s': trigger_rate,
            'mean_anode_current_A': pe_rate * spe_charge_C,
        })
results = pd.DataFrame(rows)
results

In [ ]:
# Show the simulated photoelectron distribution for photon-positive S2 events.
central_s2_pe = sampled_pe[('central', 'S2')]
incident_s2 = events['weighted_S2_photons_at_plane'].to_numpy()
positive = incident_s2 > 0

print(f'S2 photon-positive Monte Carlo events: {positive.sum():,}')
print(f'Photoelectron-positive events in this sampled realization: {(central_s2_pe > 0).sum():,}')
if np.any(central_s2_pe > 0):
    print('Nonzero S2 PE percentiles:', np.percentile(central_s2_pe[central_s2_pe > 0], [50, 90, 95, 99]))

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.arange(central_s2_pe[positive].max() + 2) - 0.5
ax.hist(central_s2_pe[positive], bins=bins, color='tab:blue', alpha=0.8)
ax.set(xlabel='Detected photoelectrons per photon-positive S2 event', ylabel='Simulated events',
       title='Predicted S2 photoelectron distribution: central extrapolation')
ax.set_yscale('log')
ax.grid(alpha=0.25)
plt.show()

## Illustrative oscilloscope pulse

The CSV does not contain individual photon arrival times, so this is not yet a physical S1/S2 timing waveform. It places all photoelectrons together in a Gaussian SPE template to show the charge-to-voltage scale. Replace this template with the measured SPE waveform and simulated arrival times when those are available.

In [ ]:
# Select a representative central-case S2 event near the median nonzero PE count.
nonzero_pe = central_s2_pe[central_s2_pe > 0]
representative_pe = int(max(1, np.median(nonzero_pe))) if len(nonzero_pe) else 1
time_ns = np.linspace(-30, 70, 1001)
sigma_ns = SPE_FWHM_NS / 2.35482

# A unit-area Gaussian in 1/ns; multiplying by mV*ns gives millivolts.
unit_template = np.exp(-0.5 * (time_ns / sigma_ns) ** 2) / (np.sqrt(2 * np.pi) * sigma_ns)
ideal_signal_mV = -representative_pe * spe_area_mV_ns * unit_template
noisy_signal_mV = ideal_signal_mV + rng.normal(0.0, BASELINE_RMS_MV, len(time_ns))
threshold_mV = -TRIGGER_SIGMA * BASELINE_RMS_MV

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(time_ns, noisy_signal_mV, lw=1, label='Illustrative signal + scope noise')
ax.plot(time_ns, ideal_signal_mV, lw=2, label='Ideal summed PMT pulse')
ax.axhline(threshold_mV, color='tab:red', ls='--', label=f'-{TRIGGER_SIGMA:g} sigma threshold')
ax.set(xlabel='Time (ns)', ylabel='Voltage across 50 ohm (mV)',
       title=f'Illustrative simultaneous {representative_pe}-PE pulse at {PMT_VOLTAGE_V:.0f} V')
ax.grid(alpha=0.25)
ax.legend()
plt.show()

print(f'Ideal peak: {ideal_signal_mV.min():.3f} mV')
print(f'Nominal threshold: {threshold_mV:.3f} mV')

## Interpretation checklist

1. The Ba-133 356 keV branching fraction is already included in `GAMMA_RATE_HZ`; do not multiply the final rates by 0.6205 again.
2. Replace `WINDOW_TRANSMISSION = 1` with the physical PMT entrance-window transmission at 128 nm.
3. Replace `COLLECTION_EFFICIENCY = 1` if a first-dynode collection efficiency is available.
4. The datasheet curve gives a typical gain, not the measured gain of this individual tube.
5. A real trigger prediction requires photon arrival times and the measured SPE pulse template.
6. Compare the predicted trigger rate with a live-time-corrected PMT dark-count rate, not merely the number of negative-trigger waveforms.